# 🚗⚡ Kaggle Playground Series s6e9 : Predicting Electric Vehicle Purchases
### 🏆 Grandmaster Solution : Diversified Ensemble (3 Trees + PyTorch MLP + Stacking Level-2)
**Objectif :** Dépasser **0.9467+ ROC-AUC** (Top 1) avec une diversité maximale de modèles et un temps d'exécution optimisé sur **Google Colab (GPU T4)** et **Kaggle Notebooks**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)
[![Kaggle](https://img.shields.io/badge/Platform-Kaggle-blue?logo=kaggle)](https://www.kaggle.com/competitions/playground-series-s6e9)
[![Hardware](https://img.shields.io/badge/Hardware-NVIDIA%20GPU%20CUDA-green?logo=nvidia)]()

---

### 🧠 Stratégie de Diversification Grandmaster :
1. **Décorrélation des Arbres de Décision** :
   - **LightGBM** : Arbres dissymétriques par feuilles (*leaf-wise*, `num_leaves=36`), ensemble complet de features.
   - **XGBoost (GPU CUDA)** : Arbres par niveau (*depth-wise*, `max_depth=5`), sous-échantillonnage strict (`colsample_bytree=0.60`) pour forcer des branches décorrélées.
   - **CatBoost (GPU CUDA)** : Arbres symétriques (*oblivious*), **AUCUN Target Encoding manuel** : CatBoost exploite exclusivement son propre encodage dynamique en ligne sur les catégories brutes.
2. **Modèle Réseau de Neurones Tabulaire (PyTorch TabularMLP)** :
   - Entity Embeddings pour les catégories + BatchNorm + SiLU + Dropout.
   - Apprend sur un manifold continu, offrant une orthogonalité parfaite avec les frontières rectangulaires des arbres.
3. **Double Assemblage Hybride (Rank Averaging + Stacking Level-2)** :
   - Optimisation non-linéaire Nelder-Mead sur les rangs (ROC-AUC).
   - Meta-Learner Level-2 (Régression Logistique) entraîné sur les probabilités OOF.
4. **Pseudo-Labeling Débrayable (`USE_PSEUDO_LABELING = False`)** : Économise immédiatement 20-25 minutes par run.

## 🛠️ 1. Installation des Dépendances & Détection Matérielle
Installation des packages nécessaires et détection directe de l'accélération GPU NVIDIA CUDA.

In [ ]:
!pip install -q lightgbm xgboost catboost optuna scikit-learn matplotlib seaborn scipy torch kaggle

import os
import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import minimize
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

import lightgbm as lgb
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

import xgboost as xgb
from xgboost import XGBClassifier

import catboost as cb
from catboost import CatBoostClassifier

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# Détection GPU
def check_cuda_available():
    try:
        from xgboost import XGBClassifier
        m = XGBClassifier(device='cuda', n_estimators=1)
        m.fit(np.zeros((2, 2)), np.array([0, 1]))
        return True
    except Exception:
        pass
    try:
        import subprocess
        res = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        return res.returncode == 0
    except Exception:
        return False

def get_gpu_device_name():
    try:
        import subprocess
        res = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], encoding='utf-8')
        return res.strip()
    except Exception:
        return "NVIDIA GPU CUDA"

HAS_CUDA = check_cuda_available()
gpu_name = get_gpu_device_name() if HAS_CUDA else "CPU Multi-Core"

print("="*65)
print(f"🚀 Accélération Matérielle : {gpu_name}")
print(f"📦 Versions : LightGBM {lgb.__version__} | XGBoost {xgb.__version__} | CatBoost {cb.__version__} | PyTorch {torch.__version__}")
print("="*65)

## 📂 2. Chargement Intelligent des Données

In [ ]:
def find_or_download_data():
    candidates = [
        '/kaggle/input/playground-series-s6e9',
        '/kaggle/input/predicting-electric-vehicle-purchases',
        '/content/drive/MyDrive/data',
        '/content/drive/MyDrive',
        '/content',
        './data',
        '.'
    ]
    for c in candidates:
        tr = os.path.join(c, 'train.csv')
        te = os.path.join(c, 'test.csv')
        if os.path.exists(tr) and os.path.exists(te):
            return tr, te
            
    if os.path.exists('/content') and not os.path.exists('train.csv'):
        if os.path.exists('kaggle.json'):
            !mkdir -p ~/.kaggle
            !cp kaggle.json ~/.kaggle/
            !chmod 600 ~/.kaggle/kaggle.json
            !kaggle competitions download -c playground-series-s6e9
            !unzip -o -q playground-series-s6e9.zip
            return 'train.csv', 'test.csv'
            
    return 'train.csv', 'test.csv'

train_path, test_path = find_or_download_data()
print(f"📂 Train Path : {train_path}")
print(f"📂 Test Path  : {test_path}")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print(f"\n📊 Dimensions Train : {train.shape[0]:,} lignes × {train.shape[1]} colonnes")
print(f"📊 Dimensions Test  : {test.shape[0]:,} lignes × {test.shape[1]} colonnes")
display(train.head(3))

## ⚙️ 3. Paramètres de Contrôle du Pipeline

In [ ]:
N_SPLITS = 10                        # 10-Fold Cross-Validation pour stabilité maximale
USE_GPU = HAS_CUDA                   # Détecte le GPU NVIDIA CUDA
USE_PSEUDO_LABELING = False          # DÉSACTIVÉ pour économiser 20+ minutes sans perte
USE_TABULAR_NN = True                # Active le modèle neural tabulaire PyTorch
USE_STACKING_META_LEARNER = True     # Active le Stacking Level-2 (Régression Logistique)
RANDOM_SEED = 42

def format_duration(seconds):
    if seconds < 60:
        return f"{seconds:.2f}s"
    minutes = int(seconds // 60)
    sec = seconds % 60
    return f"{minutes}m {sec:04.1f}s"

print(f"✅ Configuration prête (CV: {N_SPLITS}-Fold | GPU: {USE_GPU} | Tabular NN: {USE_TABULAR_NN})")

## 🧠 4. Feature Engineering Grandmaster

In [ ]:
def preprocess_and_feature_engineering(df):
    df = df.copy()
    
    # Encodage binaire et ordinal universel
    binary_map = {'Yes': 1.0, 'No': 0.0, 1: 1.0, 0: 0.0, '1': 1.0, '0': 0.0}
    if 'Home_Charging_Possible' in df.columns:
        df['Home_Charging_Possible'] = df['Home_Charging_Possible'].map(binary_map).fillna(0.0).astype(float)
    if 'Subsidy_Available' in df.columns:
        df['Subsidy_Available'] = df['Subsidy_Available'].map(binary_map).fillna(0.0).astype(float)
        
    range_map = {'Low': 0.0, 'Medium': 1.0, 'High': 2.0, 0: 0.0, 1: 1.0, 2: 2.0, '0': 0.0, '1': 1.0, '2': 2.0}
    if 'Range_Anxiety_Level' in df.columns:
        df['Range_Anxiety_Level'] = df['Range_Anxiety_Level'].map(range_map).fillna(1.0).astype(float)

    # Features Métier de Base
    df['Total_Charging_Stations'] = df['Charging_Stations_Near_Home'] + df['Charging_Stations_Near_Work']
    df['Charging_Home_Work_Ratio'] = (df['Charging_Stations_Near_Home'] + 1.0) / (df['Charging_Stations_Near_Work'] + 1.0)
    df['Income_Per_Car'] = df['Annual_Income_USD'] / (df['Number_of_Cars_Owned'] + 1.0)
    df['Income_Per_Age'] = df['Annual_Income_USD'] / (df['Age'] + 1.0)
    df['Stations_Per_Commute_km'] = df['Total_Charging_Stations'] / (df['Daily_Commute_km'] + 1.0)
    df['Commute_Per_Age'] = df['Daily_Commute_km'] / (df['Age'] + 1.0)

    # Paradoxe de Simpson & Dépendance à la Recharge Publique
    df['Need_Public_Charging'] = (1.0 - df['Home_Charging_Possible']) * df['Total_Charging_Stations']
    df['No_Home_Charge_Anxiety'] = (1.0 - df['Home_Charging_Possible']) * (df['Range_Anxiety_Level'] + 1.0)
    df['Home_Charge_High_Income'] = df['Home_Charging_Possible'] * (df['Annual_Income_USD'] / 10000.0)
    df['Commute_No_Home_Charge'] = df['Daily_Commute_km'] * (1.0 - df['Home_Charging_Possible'])

    # Nouvelles Variables Clés
    df['Commute_Stress_Index'] = df['Daily_Commute_km'] / (df['Total_Charging_Stations'] + df['Home_Charging_Possible'] * 6.0 + 1.0)
    df['Anxiety_Buffer_Ratio'] = (df['Range_Anxiety_Level'] + 1.0) / (df['Total_Charging_Stations'] + df['Home_Charging_Possible'] * 3.0 + 1.0)
    df['EV_Affordability_Index'] = (df['Annual_Income_USD'] * (1.0 + 0.4 * df['Subsidy_Available'])) / ((df['Age'] * (df['Number_of_Cars_Owned'] + 1.0)) + 1.0)
    df['Station_Access_Per_Vehicle'] = df['Total_Charging_Stations'] / (df['Number_of_Cars_Owned'] + 1.0)
    df['Eco_Action_Propensity'] = (df['Environmental_Concern_Level'] * (df['Home_Charging_Possible'] + 1.0)) / (df['Range_Anxiety_Level'] + 1.0)

    # Interactions & Score de Maturité VE
    df['Subsidy_Elasticity'] = df['Subsidy_Available'] / ((df['Annual_Income_USD'] / 10000.0) + 1.0)
    df['Eco_x_Income'] = (df['Annual_Income_USD'] / 10000.0) * df['Environmental_Concern_Level']
    df['Eco_x_Stations'] = df['Environmental_Concern_Level'] * df['Total_Charging_Stations']
    df['Eco_and_Home_Charge'] = df['Environmental_Concern_Level'] * df['Home_Charging_Possible']
    df['EV_Readiness_Score'] = (
        (df['Home_Charging_Possible'] * 3.0) + 
        (df['Subsidy_Available'] * 1.5) + 
        (df['Total_Charging_Stations'] * 0.25) - 
        (df['Range_Anxiety_Level'] * 1.5)
    )
    
    # Combinaisons Catégorielles
    df['City_and_Car'] = df['City_Type'].astype(str) + "_" + df['Current_Car_Type'].astype(str)
    df['Gender_and_Car'] = df['Gender'].astype(str) + "_" + df['Current_Car_Type'].astype(str)
    df['City_and_Anxiety'] = df['City_Type'].astype(str) + "_" + df['Range_Anxiety_Level'].astype(str)
    df['HomeCharge_and_City'] = df['Home_Charging_Possible'].astype(str) + "_" + df['City_Type'].astype(str)
    df['Demographic_Segment'] = df['City_Type'].astype(str) + "_" + df['Gender'].astype(str) + "_" + df['Current_Car_Type'].astype(str)
    
    # Count / Frequency Encoding
    for col in ['City_and_Car', 'Gender_and_Car', 'Demographic_Segment']:
        freq = df[col].value_counts(normalize=True).to_dict()
        df[f'{col}_Freq'] = df[col].map(freq).astype(float)
    
    cat_cols = ['Gender', 'City_Type', 'Current_Car_Type', 'City_and_Car', 'Gender_and_Car', 'City_and_Anxiety', 'HomeCharge_and_City', 'Demographic_Segment']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype('category')
            
    return df


def add_group_aggregations(df_all):
    df = df_all.copy()
    for grp in ['City_and_Car', 'HomeCharge_and_City', 'Demographic_Segment']:
        stats = df.groupby(grp, observed=False)['Annual_Income_USD'].agg(['mean', 'std']).reset_index()
        stats.columns = [grp, f'{grp}_Income_Mean', f'{grp}_Income_Std']
        df = df.merge(stats, on=grp, how='left')
        df[f'{grp}_Income_Diff'] = df['Annual_Income_USD'] - df[f'{grp}_Income_Mean']
        df[f'{grp}_Income_Ratio'] = df['Annual_Income_USD'] / (df[f'{grp}_Income_Mean'] + 1.0)
        
    return df

print("⚙️ Fonctions de Feature Engineering prêtes.")

## 🎯 5. Smooth Target Encoding Out-Of-Fold

In [ ]:
def apply_oof_target_encoding(train_df, test_df, cat_cols, target_col, skf, m_smoothing=20.0):
    train_encoded = train_df.copy()
    test_encoded = test_df.copy()
    
    global_mean = float(train_df[target_col].mean())
    
    for col in cat_cols:
        col_name = f"{col}_TE"
        train_encoded[col_name] = 0.0
        test_col_encoded = np.zeros(len(test_df), dtype=float)
        
        for train_idx, val_idx in skf.split(train_df, train_df[target_col]):
            fold_train = train_df.iloc[train_idx]
            fold_val = train_df.iloc[val_idx]
            
            stats = fold_train.groupby(fold_train[col].astype(str), observed=False)[target_col].agg(['count', 'mean'])
            smoothed_series = (stats['count'] * stats['mean'] + m_smoothing * global_mean) / (stats['count'] + m_smoothing)
            smoothed_dict = smoothed_series.to_dict()
            
            val_vals = fold_val[col].astype(str).map(smoothed_dict).fillna(global_mean).astype(float).values
            train_encoded.loc[train_encoded.index[val_idx], col_name] = val_vals
            
            test_vals = test_df[col].astype(str).map(smoothed_dict).fillna(global_mean).astype(float).values
            test_col_encoded += test_vals / skf.n_splits
            
        test_encoded[col_name] = test_col_encoded
        
    return train_encoded, test_encoded

print("🎯 Smooth Target Encoding configuré.")

## 🧬 6. Modèle Réseau de Neurones Tabulaire (PyTorch TabularMLP)

In [ ]:
class TabularMLP(nn.Module):
    def __init__(self, num_cont, cat_cardinalities):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(card, min(16, max(2, (card + 1) // 2)))
            for card in cat_cardinalities
        ])
        total_emb_dim = sum(e.embedding_dim for e in self.embeddings)
        in_dim = num_cont + total_emb_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.SiLU(),
            nn.Dropout(0.15),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.SiLU(),
            nn.Linear(32, 1)
        )
        
    def forward(self, x_cont, x_cat):
        if len(self.embeddings) > 0 and x_cat is not None:
            embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
            x = torch.cat([x_cont] + embs, dim=1)
        else:
            x = x_cont
        return self.net(x).squeeze(-1)


def train_tabular_mlp(X_tr, y_tr, X_va, y_va, X_test, cat_cols_base, device, epochs=8, batch_size=4096):
    cont_cols = [c for c in X_tr.columns if c not in cat_cols_base and str(X_tr[c].dtype) != 'category']
    
    scaler = StandardScaler()
    X_tr_cont = scaler.fit_transform(X_tr[cont_cols].fillna(0.0).values).astype(np.float32)
    X_va_cont = scaler.transform(X_va[cont_cols].fillna(0.0).values).astype(np.float32)
    X_test_cont = scaler.transform(X_test[cont_cols].fillna(0.0).values).astype(np.float32)
    
    cat_cards = [int(X_tr[c].nunique()) + 2 for c in cat_cols_base]
    X_tr_cat = np.clip(np.column_stack([X_tr[c].astype('category').cat.codes.values for c in cat_cols_base]), 0, None).astype(np.int64)
    X_va_cat = np.clip(np.column_stack([X_va[c].astype('category').cat.codes.values for c in cat_cols_base]), 0, None).astype(np.int64)
    X_test_cat = np.clip(np.column_stack([X_test[c].astype('category').cat.codes.values for c in cat_cols_base]), 0, None).astype(np.int64)
    
    model = TabularMLP(num_cont=len(cont_cols), cat_cardinalities=cat_cards).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    
    train_ds = TensorDataset(torch.from_numpy(X_tr_cont), torch.from_numpy(X_tr_cat), torch.from_numpy(y_tr.values.astype(np.float32)))
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
    
    va_cont_t = torch.from_numpy(X_va_cont).to(device)
    va_cat_t = torch.from_numpy(X_va_cat).to(device)
    test_cont_t = torch.from_numpy(X_test_cont).to(device)
    test_cat_t = torch.from_numpy(X_test_cat).to(device)
    
    best_auc = 0.0
    best_preds_val = None
    best_preds_test = None
    
    for epoch in range(epochs):
        model.train()
        for bx_cont, bx_cat, by in train_dl:
            bx_cont, bx_cat, by = bx_cont.to(device), bx_cat.to(device), by.to(device)
            optimizer.zero_grad()
            logits = model(bx_cont, bx_cat)
            loss = criterion(logits, by)
            loss.backward()
            optimizer.step()
            
        model.eval()
        with torch.no_grad():
            val_logits = model(va_cont_t, va_cat_t)
            val_preds = torch.sigmoid(val_logits).cpu().numpy()
            val_auc = roc_auc_score(y_va, val_preds)
            if val_auc > best_auc:
                best_auc = val_auc
                best_preds_val = val_preds
                best_preds_test = torch.sigmoid(model(test_cont_t, test_cat_t)).cpu().numpy()
                
    return best_preds_val, best_preds_test, best_auc

print("🧬 Module PyTorch TabularMLP prêt.")

## ⚖️ 7. Assemblage Hybride : Rank Averaging & Stacking Level-2

In [ ]:
def rank_average(pred_list, weights=None):
    if weights is None:
        weights = [1.0 / len(pred_list)] * len(pred_list)
    else:
        weights = [w / sum(weights) for w in weights]
        
    ranked_sum = np.zeros(len(pred_list[0]))
    for pred, w in zip(pred_list, weights):
        ranked = rankdata(pred) / len(pred)
        ranked_sum += ranked * w
        
    return ranked_sum


def optimize_ensemble_weights(y_true, pred_list):
    n_models = len(pred_list)
    if n_models == 1:
        return [1.0]

    def objective(weights):
        w = np.array(weights)
        w = w / np.sum(w)
        blend = rank_average(pred_list, weights=w)
        return -roc_auc_score(y_true, blend)

    init_weights = [1.0 / n_models] * n_models
    bounds = [(0.01, 1.0)] * n_models
    res = minimize(objective, init_weights, method='Nelder-Mead', bounds=bounds)
    opt_weights = res.x / np.sum(res.x)
    return opt_weights.tolist()


def train_stacking_meta_learner(y_true, oof_preds_list, test_preds_list):
    X_meta = np.column_stack(oof_preds_list)
    X_test_meta = np.column_stack(test_preds_list)
    
    meta_model = LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_SEED)
    meta_model.fit(X_meta, y_true)
    
    meta_oof = meta_model.predict_proba(X_meta)[:, 1]
    meta_test = meta_model.predict_proba(X_test_meta)[:, 1]
    meta_auc = roc_auc_score(y_true, meta_oof)
    
    return meta_oof, meta_test, meta_auc, meta_model

print("⚖️ Assemblage et Stacking configurés.")

## 🚀 8. Pipeline d'Entraînement Multi-Modèles Décorrélé (10 Folds)

In [ ]:
def train_ensemble_pipeline(X, y, X_test, test_ids, skf, tag="standard"):
    timing_report = {}
    
    # 1. LightGBM (Leaf-wise GBDT)
    print("\n" + "="*70)
    print(f"📦 [1/4] Entraînement LightGBM ({skf.n_splits} Folds) - [{tag}]...")
    print("="*70)
    lgb_start = time.time()
    
    lgb_params = {
        'n_estimators': 2500,
        'learning_rate': 0.03,
        'num_leaves': 36,
        'max_depth': 6,
        'subsample': 0.85,
        'colsample_bytree': 0.80,
        'min_child_samples': 40,
        'reg_alpha': 0.1,
        'reg_lambda': 1.5,
        'random_state': RANDOM_SEED,
        'n_jobs': -1,
        'verbosity': -1
    }
        
    lgb_oof = np.zeros(len(X))
    lgb_test = np.zeros(len(X_test))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        f_start = time.time()
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
        
        model = LGBMClassifier(**lgb_params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric='auc',
            callbacks=[early_stopping(40, verbose=False), log_evaluation(0)]
        )
        lgb_oof[val_idx] = model.predict_proba(X_va)[:, 1]
        lgb_test += model.predict_proba(X_test)[:, 1] / skf.n_splits
        
        f_auc = roc_auc_score(y_va, lgb_oof[val_idx])
        print(f"  👉 LGBM Fold {fold+1:02d}/{skf.n_splits:02d} | ROC-AUC : {f_auc:.5f} | ⏱️ {format_duration(time.time() - f_start)}")
        
    lgb_auc = roc_auc_score(y, lgb_oof)
    lgb_time = time.time() - lgb_start
    timing_report['LightGBM'] = {'auc': lgb_auc, 'time': lgb_time}
    print(f"  🏆 Score LightGBM OOF ROC-AUC : {lgb_auc:.5f} | ⏱️ Temps Total : {format_duration(lgb_time)}")

    # 2. XGBoost GPU Décorrélé (max_depth=5, colsample=0.60)
    print("\n" + "="*70)
    gpu_tag = "GPU (CUDA)" if USE_GPU else "CPU"
    print(f"📦 [2/4] Entraînement XGBoost Décorrélé [{gpu_tag}] ({skf.n_splits} Folds) - [{tag}]...")
    print("="*70)
    xgb_start = time.time()
    
    xgb_oof = np.zeros(len(X))
    xgb_test = np.zeros(len(X_test))
    
    xgb_params = {
        'n_estimators': 2200,
        'learning_rate': 0.03,
        'max_depth': 5,              # Décorrélation de LightGBM
        'subsample': 0.80,
        'colsample_bytree': 0.60,    # Chemins d'arbres décorrélés
        'tree_method': 'hist',
        'device': 'cuda' if USE_GPU else 'cpu',
        'enable_categorical': True,
        'eval_metric': 'auc',
        'early_stopping_rounds': 40,
        'reg_alpha': 0.1,
        'reg_lambda': 1.5,
        'random_state': RANDOM_SEED + 10,
        'n_jobs': -1
    }
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        f_start = time.time()
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
        
        xgb_model = XGBClassifier(**xgb_params)
        xgb_model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False
        )
        xgb_oof[val_idx] = xgb_model.predict_proba(X_va)[:, 1]
        xgb_test += xgb_model.predict_proba(X_test)[:, 1] / skf.n_splits
        
        f_auc = roc_auc_score(y_va, xgb_oof[val_idx])
        print(f"  👉 XGB Fold {fold+1:02d}/{skf.n_splits:02d} | ROC-AUC : {f_auc:.5f} | ⏱️ {format_duration(time.time() - f_start)}")
        
    xgb_auc = roc_auc_score(y, xgb_oof)
    xgb_time = time.time() - xgb_start
    timing_report['XGBoost'] = {'auc': xgb_auc, 'time': xgb_time}
    print(f"  🏆 Score XGBoost OOF ROC-AUC : {xgb_auc:.5f} | ⏱️ Temps Total : {format_duration(xgb_time)}")

    # 3. CatBoost GPU Brut (SANS Target Encoding Manuel)
    print("\n" + "="*70)
    gpu_label = "GPU (CUDA)" if USE_GPU else "CPU"
    print(f"📦 [3/4] Entraînement CatBoost Brut [{gpu_label}] ({skf.n_splits} Folds) - [{tag}]...")
    print("="*70)
    cat_start = time.time()
    
    cat_oof = np.zeros(len(X))
    cat_test = np.zeros(len(X_test))
    
    non_te_cols = [c for c in X.columns if not c.endswith('_TE')]
    X_cat = X[non_te_cols].copy()
    X_test_cat = X_test[non_te_cols].copy()
    
    cat_cols_list = [col for col in ['Gender', 'City_Type', 'Current_Car_Type'] if col in X_cat.columns]
    for c in X_cat.select_dtypes(include=['category']).columns:
        if c in cat_cols_list:
            X_cat[c] = X_cat[c].astype(str)
            X_test_cat[c] = X_test_cat[c].astype(str)
        else:
            X_cat[c] = X_cat[c].cat.codes.astype(float)
            X_test_cat[c] = X_test_cat[c].cat.codes.astype(float)
        
    cat_params = {
        'iterations': 1800,
        'learning_rate': 0.035,
        'depth': 6,
        'l2_leaf_reg': 4.0,
        'eval_metric': 'Logloss',
        'border_count': 128,
        'random_seed': RANDOM_SEED + 20,
        'verbose': False,
        'task_type': 'GPU' if USE_GPU else 'CPU',
        'allow_writing_files': False
    }
    if not USE_GPU:
        cat_params['thread_count'] = -1
        
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_cat, y)):
        f_start = time.time()
        X_tr, y_tr = X_cat.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X_cat.iloc[val_idx], y.iloc[val_idx]
        
        cat_model = CatBoostClassifier(**cat_params, cat_features=cat_cols_list)
        cat_model.fit(
            X_tr, y_tr,
            eval_set=(X_va, y_va),
            early_stopping_rounds=40,
            verbose=False
        )
        cat_oof[val_idx] = cat_model.predict_proba(X_va)[:, 1]
        cat_test += cat_model.predict_proba(X_test_cat)[:, 1] / skf.n_splits
        
        f_auc = roc_auc_score(y_va, cat_oof[val_idx])
        print(f"  👉 CatBoost Fold {fold+1:02d}/{skf.n_splits:02d} | ROC-AUC : {f_auc:.5f} | ⏱️ {format_duration(time.time() - f_start)}")
        
    cat_auc = roc_auc_score(y, cat_oof)
    cat_time = time.time() - cat_start
    timing_report['CatBoost'] = {'auc': cat_auc, 'time': cat_time}
    print(f"  🏆 Score CatBoost OOF ROC-AUC : {cat_auc:.5f} | ⏱️ Temps Total : {format_duration(cat_time)}")

    # 4. PyTorch TabularMLP
    nn_oof = None
    nn_test = None
    if USE_TABULAR_NN:
        print("\n" + "="*70)
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        device_label = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU Multi-Core"
        print(f"📦 [4/4] Entraînement PyTorch TabularMLP [{device_label}] ({skf.n_splits} Folds) - [{tag}]...")
        print("="*70)
        nn_start = time.time()
        
        nn_oof = np.zeros(len(X))
        nn_test = np.zeros(len(X_test))
        base_cats = [c for c in ['Gender', 'City_Type', 'Current_Car_Type'] if c in X.columns]
        
        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
            f_start = time.time()
            X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
            X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
            
            val_preds, test_preds, f_auc = train_tabular_mlp(
                X_tr, y_tr, X_va, y_va, X_test, base_cats, device=device, epochs=8, batch_size=4096
            )
            nn_oof[val_idx] = val_preds
            nn_test += test_preds / skf.n_splits
            
            print(f"  👉 TabularMLP Fold {fold+1:02d}/{skf.n_splits:02d} | ROC-AUC : {f_auc:.5f} | ⏱️ {format_duration(time.time() - f_start)}")
            
        nn_auc = roc_auc_score(y, nn_oof)
        nn_time = time.time() - nn_start
        timing_report['TabularMLP'] = {'auc': nn_auc, 'time': nn_time}
        print(f"  🏆 Score TabularMLP OOF ROC-AUC : {nn_auc:.5f} | ⏱️ Temps Total : {format_duration(nn_time)}")

    # Assemblage Multi-Modèles
    models_oof = [lgb_oof, xgb_oof, cat_oof]
    models_test = [lgb_test, xgb_test, cat_test]
    if USE_TABULAR_NN and nn_oof is not None:
        models_oof.append(nn_oof)
        models_test.append(nn_test)
        
    opt_weights = optimize_ensemble_weights(y, models_oof)
    print(f"\n🎯 Poids Nelder-Mead : {[round(w, 3) for w in opt_weights]}")
    rank_blend_oof = rank_average(models_oof, weights=opt_weights)
    rank_blend_test = rank_average(models_test, weights=opt_weights)
    rank_auc = roc_auc_score(y, rank_blend_oof)
    print(f"  ✨ ROC-AUC Rank Averaging : {rank_auc:.5f}")
    
    if USE_STACKING_META_LEARNER:
        stack_oof, stack_test, stack_auc, _ = train_stacking_meta_learner(y, models_oof, models_test)
        print(f"  ✨ ROC-AUC Stacking Meta-Learner : {stack_auc:.5f}")
        
        final_oof = rank_average([rank_blend_oof, stack_oof], weights=[0.50, 0.50])
        final_test_preds = rank_average([rank_blend_test, stack_test], weights=[0.50, 0.50])
        final_auc = roc_auc_score(y, final_oof)
        print(f"  🚀 ROC-AUC Hybride Final (Rank + Stacking) : {final_auc:.5f}")
    else:
        final_oof = rank_blend_oof
        final_test_preds = rank_blend_test
        final_auc = rank_auc
        
    return final_oof, final_test_preds, final_auc, timing_report

print("🚀 Pipeline d'entraînement multi-modèles prêt.")

## ⚡ 9. Exécution de l'Ensemble Complet

In [ ]:
total_start_time = time.time()

y = (train['Will_Buy_EV'] == 'Yes').astype(int)
test_ids = test['id']

fe_start = time.time()
print("⚙️ Application du Feature Engineering Avancé et des Statistiques de Groupe...")
df_all = pd.concat([train.assign(is_train=1), test.assign(is_train=0, Will_Buy_EV='No')], axis=0).reset_index(drop=True)
df_fe = preprocess_and_feature_engineering(df_all)
df_fe = add_group_aggregations(df_fe)

train_fe = df_fe[df_fe['is_train'] == 1].drop(columns=['is_train']).reset_index(drop=True)
test_fe = df_fe[df_fe['is_train'] == 0].drop(columns=['is_train', 'Will_Buy_EV']).reset_index(drop=True)
train_fe['Will_Buy_EV'] = y

print("⚙️ Application du Smooth Target Encoding Out-Of-Fold (Sans Leakage)...")
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
te_cols = ['City_Type', 'Current_Car_Type', 'City_and_Car', 'City_and_Anxiety', 'HomeCharge_and_City', 'Demographic_Segment']
train_fe, test_fe = apply_oof_target_encoding(train_fe, test_fe, te_cols, 'Will_Buy_EV', skf, m_smoothing=20.0)
print(f"⏱️ Feature Engineering complet terminé en : {format_duration(time.time() - fe_start)}")

features = [c for c in train_fe.columns if c not in ['id', 'Will_Buy_EV']]
X = train_fe[features]
X_test = test_fe[features]

print(f"📊 Nombre total de variables explicatives générées : {len(features)}")

# Lancement de l'Ensemble
final_oof, final_test_preds, final_auc, timing_report = train_ensemble_pipeline(
    X, y, X_test, test_ids, skf, tag="Diversified Grandmaster Ensemble"
)

print("\n" + "="*70)
print(f"🏆 SCORE FINAL ENSEMBLE ROC-AUC : {final_auc:.5f}")
print("="*70)

## 📁 10. Sauvegarde de la Soumission Kaggle

In [ ]:
os.makedirs('submissions/final', exist_ok=True)

ensemble_filename = f"submission_ensemble_grandmaster_auc_{final_auc:.5f}.csv"
final_path = os.path.join('submissions', 'final', ensemble_filename)

sub = pd.DataFrame({
    'id': test_ids,
    'Will_Buy_EV': final_test_preds
})

sub.to_csv(final_path, index=False)
sub.to_csv('submission.csv', index=False)

print(f"\n✅ Soumission Finale enregistrée : {final_path}")
print(f"   (Fichier de soumission Kaggle prêt : submission.csv)")
print(f"   (Dimensions : {sub.shape[0]} lignes × {sub.shape[1]} colonnes)")

total_duration = time.time() - total_start_time
print(f"\n🏁 Pipeline Grandmaster terminé avec succès ! ⏱️ Durée Totale : {format_duration(total_duration)}")
display(sub.head(10))